In [ ]:
# Instalación de dependencias
import sys
!{sys.executable} -m pip install aiohttp -q
print("✅ aiohttp instalado")

In [ ]:
#Descargar capítulos MangaDex vía API oficial
import aiohttp
import asyncio
import os
import re
import json

#Configuración
CHAPTER_ID = "6688db57-2b69-4bb5-a611-30ca37487a27" #Aquí va el ID del capítulo que quieres descargar
CARPETA    = "./raws/prueba_jupyter"
CALIDAD    = "data"        # "data" = alta, "dataSaver" = comprimida
DELAY      = 0.3           

os.makedirs(CARPETA, exist_ok=True)

API_BASE = "https://api.mangadex.org" #Reglas de uso de la api: https://api.mangadex.org/docs/
HEADERS = {
    "User-Agent": "MiDescargadorPersonal/1.0 (contacto: tu@email.com)"
}


async def obtener_info_capitulo(session, chapter_id):
    """Consulta /at-home/server/{id} y devuelve los datos del capítulo."""
    url = f"{API_BASE}/at-home/server/{chapter_id}"
    async with session.get(url, headers=HEADERS) as r:
        if r.status == 404:
            raise ValueError(f" Capítulo {chapter_id} no encontrado (404)")
        if r.status == 429:
            raise RuntimeError(" Rate limit alcanzado, espera un momento")
        if r.status != 200:
            raise RuntimeError(f" Error API: HTTP {r.status} - {await r.text()}")
        return await r.json()


async def descargar_imagen(session, url, ruta_destino):
    """Descarga una sola imagen."""
    async with session.get(url, headers=HEADERS) as r:
        if r.status == 200:
            with open(ruta_destino, "wb") as f:
                f.write(await r.read())
            return True
        return False


async def descargar_capitulo(chapter_id, carpeta, calidad="data"):
    """Descarga todas las páginas de un capítulo."""
    async with aiohttp.ClientSession() as session:
        # 1)Info del capítulo
        print(f"- Consultando API para capítulo {chapter_id}...")
        info = await obtener_info_capitulo(session, chapter_id)

        base_url   = info["baseUrl"]
        chapter    = info["chapter"]
        hash_cap   = chapter["hash"]
        paginas    = chapter[calidad]      # lista de nombres de archivo

        print(f"- Capítulo: {chapter.get('title') or '(sin título)'}")
        print(f"- Páginas: {len(paginas)}")
        print(f"- Servidor: {base_url}")
        print(f"- Calidad: {'alta' if calidad == 'data' else 'comprimida'}\n")

        # 2) Detectar extensión
        exitos = 0
        fallos = 0

        for i, nombre_pagina in enumerate(paginas, start=1):
            # Construyendo la URL según la calidad
            if calidad == "data":
                url = f"{base_url}/data/{hash_cap}/{nombre_pagina}"
            else:
                url = f"{base_url}/data-saver/{hash_cap}/{nombre_pagina}"

            # Preservar la extensión original
            ext = os.path.splitext(nombre_pagina)[1] or ".jpg"
            ruta = os.path.join(carpeta, f"{i:03d}{ext}")

            try:
                ok = await descargar_imagen(session, url, ruta)
                if ok:
                    print(f"✅ {i:03d}{ext}  ({nombre_pagina[:12]}...)")
                    exitos += 1
                else:
                    print(f"⚠️ Página {i} falló")
                    fallos += 1
            except Exception as e:
                print(f"❌ Página {i}: {e}")
                fallos += 1

            await asyncio.sleep(DELAY)

        print(f"\n Descarga terminada: {exitos} OK, {fallos} fallos")
        print(f" Guardado en: {os.path.abspath(carpeta)}")
        return exitos, fallos


#Ejecución principal
await descargar_capitulo(CHAPTER_ID, CARPETA, CALIDAD)